In [1]:
import os
import shutil
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import Counter
from IPython.display import Audio, display

In [3]:
CONFIG = {
    "raw_dir"    : "/content/icbhi_dataset/raw",      # downloaded .wav + .txt
    "cycles_dir" : "/content/icbhi_dataset/cycles",   # segmented cycles go here
    "splits_file": "/content/icbhi_dataset/raw/ICBHI_challenge_train_test.txt",


    "target_sr"  : 22050,

    "class_map" : {
        (0,0): "Normal",
        (1,0): "Crackle",
        (0,1): "Wheeze",
        (1,1): "Both",
    },
    "classes"    : ["Normal", "Crackle", "Wheeze", "Both"],

    "seed" : 42,

}

In [5]:
for d in [CONFIG["raw_dir"], CONFIG["cycles_dir"]]:
    os.makedirs(d, exist_ok=True)

print("CONFIG ready.")
print(f"Target sample rate : {CONFIG['target_sr']} Hz")
print(f"Classes            : {CONFIG['classes']}")

CONFIG ready.
Target sample rate : 22050 Hz
Classes            : ['Normal', 'Crackle', 'Wheeze', 'Both']


In [6]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d vbookshelf/respiratory-sound-database \
       -p /content/icbhi_dataset/raw --unzip --quiet

print("Download complete.")

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/vbookshelf/respiratory-sound-database
License(s): unknown
Download complete.


In [8]:
def parse_annotation_file(txt_path: str) -> pd.DataFrame:
  """
    Read one ICBHI annotation .txt file and return a DataFrame.

    Args:
        txt_path: path to the .txt annotation file

    Returns:
        DataFrame with columns:
            start    → cycle start time in seconds (float)
            end      → cycle end time in seconds (float)
            crackle  → 1 if crackles present, else 0 (int)
            wheeze   → 1 if wheeze present, else 0 (int)
            label    → human-readable class string ("Normal", "Crackle", etc.)
    """

  df = pd.read_csv(
        txt_path,
        spe="\t",
        header=None,
        names=["start", "end", "crackle", "wheeze"],
    )
  df["crackle"] = df["crackle"].astype(int)
  df["wheeze"] = df["wheeze"].astype(int)


  df["label"] = df.apply(
        lambda row: CONFIG["class_map"][(row["crackle"], row["wheeze"])],
        axis=1
    )

  return df

In [ ]:
def demo_annotation_parsing(raw_dir: str):
    """Show the first annotation file we find."""
    audio_dir = os.path.join(raw_dir, "audio_and_txt_files")

    # Find any .txt file
    txt_files = list(Path(audio_dir).glob("*.txt"))
    if not txt_files:
        print("No annotation files found. Check your raw_dir path.")
        return

    sample_txt = str(txt_files[0])
    print(f"Parsing: {os.path.basename(sample_txt)}\n")

    df = parse_annotation_file(sample_txt)
    print(df.to_string(index=False))
    print(f"\nCycles in this recording: {len(df)}")
    print(f"Label counts: {df['label'].value_counts().to_dict()}")